# EDA — Financial Panel 1990–2025

**Objective:** Extract, validate, transform, analyse and persist a daily panel of 12 macro/market series covering 1990-01-02 → today.

| # | Series | Ticker/Code | Source | Freq |
|---|--------|-------------|--------|------|
| 1 | S&P 500 | `^GSPC` | Yahoo Finance | Daily |
| 2 | VIX | `VIXCLS` | FRED | Daily |
| 3 | Treasury 10Y | `DGS10` | FRED | Daily |
| 4 | Treasury 3M | `DTB3` | FRED | Daily |
| 5 | HY OAS | `BAMLH0A0HYM2` | FRED | Daily |
| 6 | US Dollar Index | `DX-Y.NYB` | Yahoo Finance | Daily |
| 7 | Gold | `GOLDAMGBD228NLBM` | FRED | Daily |
| 8 | WTI Crude Oil | `DCOILWTICO` | FRED | Daily |
| 9 | CPI | `CPIAUCSL` | FRED | Monthly |
| 10 | Fed Funds Rate | `FEDFUNDS` | FRED | Monthly |
| 11 | Industrial Production | `INDPRO` | FRED | Monthly |
| 12 | Unemployment | `UNRATE` | FRED | Monthly |

---

**Table of Contents**
1. [Setup](#1-setup)
2. [Extraction](#2-extraction)
3. [Assembly](#3-assembly)
4. [Verifications 1–5](#4-verifications)
5. [Transformations](#5-transformations)
6. [Plotly Interactive Charts](#6-plotly)
7. [Matplotlib Static Figures](#7-matplotlib)
8. [Quant Metrics](#8-quant-metrics)
9. [Statistical Analysis](#9-statistical-analysis)
10. [Output & Check 7](#10-output)

In [ ]:
# ── Install dependencies if not already available ────────────────────────────
# %pip install -q yfinance fredapi alpaca-py plotly matplotlib scipy statsmodels ipykernel

import os, io, math, warnings
from datetime import date

import boto3
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import scipy.stats as spstats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.diagnostic import acorr_ljungbox
import yfinance as yf
from fredapi import Fred

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Constants ────────────────────────────────────────────────────────────────
START = "1990-01-02"
END   = date.today().isoformat()

# Series definitions
FRED_DAILY   = ["VIXCLS", "DGS10", "DTB3", "BAMLH0A0HYM2",
                 "GOLDAMGBD228NLBM", "DCOILWTICO"]
FRED_MONTHLY = ["CPIAUCSL", "FEDFUNDS", "INDPRO", "UNRATE"]
YAHOO_TICKERS = ["^GSPC", "DX-Y.NYB"]
ALL_SERIES   = YAHOO_TICKERS + FRED_DAILY + FRED_MONTHLY   # 12 total

# Transformation groups
PRICE_COLS = ["^GSPC", "GOLDAMGBD228NLBM", "DCOILWTICO", "DX-Y.NYB"]
RATE_COLS  = ["DGS10", "DTB3", "BAMLH0A0HYM2", "FEDFUNDS"]

# ── Credentials ──────────────────────────────────────────────────────────────
FRED_API_KEY      = os.environ.get("FRED_API_KEY", "")
ALPACA_API_KEY    = os.environ.get("ALPACA_API_KEY", "")
ALPACA_SECRET_KEY = os.environ.get("ALPACA_SECRET_KEY", "")
S3_BUCKET  = os.environ.get("S3_BUCKET_NAME", "tcc-etl-bucket")
S3_PREFIX  = os.environ.get("S3_PREFIX", "data/")
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

assert FRED_API_KEY, "Set FRED_API_KEY env var first (free at fred.stlouisfed.org)"
print(f"Panel window : {START} → {END}")
print(f"S3 target    : s3://{S3_BUCKET}/{S3_PREFIX}")

## 2. Extraction

In [ ]:
# ── 2a. FRED (10 series) ─────────────────────────────────────────────────────
fred = Fred(api_key=FRED_API_KEY)

FRED_SERIES = FRED_DAILY + FRED_MONTHLY  # 10 total

fred_raw: dict[str, pd.Series] = {}
for sid in FRED_SERIES:
    s = fred.get_series(sid)
    s = s.loc[START:].rename(sid)
    fred_raw[sid] = s
    print(
        f"  {sid:28s}: {len(s):5d} obs  "
        f"{s.index[0].date()} → {s.index[-1].date()}  "
        f"NaN={int(s.isna().sum())}"
    )

print("\nFRED extraction complete.")

In [ ]:
# ── 2b. Yahoo Finance (^GSPC, DX-Y.NYB) ─────────────────────────────────────
yf_data = yf.download(
    YAHOO_TICKERS, start=START, auto_adjust=True, progress=False
)["Close"]

# Drop any tz info so indexes align later
yf_data.index = pd.to_datetime(yf_data.index).tz_localize(None)

for col in YAHOO_TICKERS:
    s = yf_data[col].dropna()
    print(
        f"  {col:12s}: {len(s):5d} obs  "
        f"{s.index[0].date()} → {s.index[-1].date()}  "
        f"NaN={int(yf_data[col].isna().sum())}"
    )

print("\nYahoo Finance extraction complete.")

In [ ]:
# ── 2c. Alpaca comparison (SPY, 2019–present, free tier) ─────────────────────
from datetime import datetime

try:
    from alpaca.data.historical import StockHistoricalDataClient
    from alpaca.data.requests import StockBarsRequest
    from alpaca.data.timeframe import TimeFrame

    _alpaca_client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)
    _req = StockBarsRequest(
        symbol_or_symbols=["SPY"],
        timeframe=TimeFrame.Day,
        start=datetime(2019, 1, 2),
    )
    _bars = _alpaca_client.get_stock_bars(_req).df
    spy_alpaca = (
        _bars["close"].droplevel(0)
        if isinstance(_bars.index, pd.MultiIndex)
        else _bars["close"]
    )
    spy_alpaca.index = pd.to_datetime(spy_alpaca.index).tz_localize(None)
    print(
        f"  Alpaca SPY: {len(spy_alpaca)} obs  "
        f"{spy_alpaca.index[0].date()} → {spy_alpaca.index[-1].date()}"
    )
    _alpaca_ok = True
except Exception as e:
    print(f"Alpaca unavailable: {e}")
    spy_alpaca = pd.Series(dtype=float)
    _alpaca_ok = False

In [ ]:
# ── 2d. Source comparison chart ──────────────────────────────────────────────
if _alpaca_ok and len(spy_alpaca) > 0:
    _ov_start = "2019-01-02"
    _gspc_ov  = yf_data["^GSPC"].loc[_ov_start:].dropna()
    _spy_ov   = spy_alpaca.loc[_ov_start:]

    _base_g = _gspc_ov.iloc[0]
    _base_s = _spy_ov.iloc[0]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=_gspc_ov.index, y=_gspc_ov / _base_g * 100,
        name="Yahoo ^GSPC (normalized)", line=dict(width=1.5),
    ))
    fig.add_trace(go.Scatter(
        x=_spy_ov.index, y=_spy_ov / _base_s * 100,
        name="Alpaca SPY (normalized)", line=dict(dash="dash", width=1.5),
    ))
    fig.update_layout(
        title="Source Comparison: Alpaca SPY vs Yahoo ^GSPC (normalized to 100 at 2019-01-02)",
        yaxis_title="Index (100 = 2019-01-02)",
        xaxis_title="Date",
        height=450,
    )
    fig.show()
    print(
        "→ Decision: Yahoo Finance used in final panel "
        "(history back to 1990; Alpaca free tier ~5 yrs only; DX-Y.NYB unavailable on Alpaca)"
    )
else:
    print("Alpaca unavailable — skipping comparison. Yahoo Finance used for all series.")

## 3. Assembly

In [ ]:
# ── 3. Build NYSE business-day spine; join all 12 series ─────────────────────

def _to_polars_col(pd_series: pd.Series, col_name: str) -> pl.DataFrame:
    """Convert a pandas Series (datetime index) to a polars Date+Float64 DataFrame."""
    df = pd.DataFrame({"date": pd_series.index, col_name: pd_series.values})
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    return (
        pl.from_pandas(df)
        .with_columns([
            pl.col("date").cast(pl.Date),
            pl.col(col_name).cast(pl.Float64),
        ])
    )

# NYSE calendar (pd.bdate_range uses freq="C" by default which is "custom business day")
spine_pd = pd.bdate_range(START, END)
spine = (
    pl.from_pandas(pd.DataFrame({"date": spine_pd}))
    .with_columns(pl.col("date").cast(pl.Date))
)

panel_raw = spine

# Join FRED series
for sid in FRED_SERIES:
    panel_raw = panel_raw.join(_to_polars_col(fred_raw[sid], sid), on="date", how="left")

# Join Yahoo Finance series
for col in YAHOO_TICKERS:
    panel_raw = panel_raw.join(
        _to_polars_col(yf_data[col].dropna(), col), on="date", how="left"
    )

print(f"panel_raw shape : {panel_raw.shape}")
print(f"Columns         : {panel_raw.columns}")
panel_raw.head(5)

## 4. Verifications

In [ ]:
# ── Check 1: Shape & Coverage ─────────────────────────────────────────────────
print("═" * 60); print("CHECK 1 — Shape & Coverage"); print("═" * 60)

# 1 date column + 12 series columns
assert panel_raw.shape[1] == 13, f"Expected 13 cols, got {panel_raw.shape[1]}"

min_date = str(panel_raw["date"].min())
assert min_date == START, f"Expected start {START}, got {min_date}"

for col in panel_raw.columns[1:]:
    n_null = panel_raw[col].null_count()
    assert n_null < panel_raw.shape[0], f"Column '{col}' is completely null!"

print(f"✓  Shape          : {panel_raw.shape}")
print(f"✓  Date range     : {panel_raw['date'].min()} → {panel_raw['date'].max()}")
print(f"✓  No fully-null columns (12 series present)")

In [ ]:
# ── Check 2: Missing Values ───────────────────────────────────────────────────
print("═" * 60); print("CHECK 2 — Missing Values"); print("═" * 60)

null_counts = {col: panel_raw[col].null_count() for col in panel_raw.columns[1:]}
null_df = pl.DataFrame({
    "series":     list(null_counts.keys()),
    "null_count": list(null_counts.values()),
    "pct":        [round(v / panel_raw.shape[0] * 100, 2) for v in null_counts.values()],
})
print(null_df)

# BAMLH0A0HYM2: nulls only before 1997 (series starts Dec 1996)
hy_post = (
    panel_raw
    .filter(pl.col("date") > pl.lit("1996-12-31").str.to_date())
    ["BAMLH0A0HYM2"]
    .null_count()
)
assert hy_post == 0, f"BAMLH0A0HYM2 has {hy_post} unexpected nulls after 1996-12-31"
print(f"\n✓  BAMLH0A0HYM2 post-1996 nulls = {hy_post}  (pre-1997 gap is expected)")

# Core equity and vol index must be fully populated
for col in ["^GSPC", "VIXCLS"]:
    n = panel_raw[col].null_count()
    assert n == 0, f"{col} has {n} nulls — expected 0"
print(f"✓  ^GSPC nulls = 0 | VIXCLS nulls = 0")

# WTI can have Friday/holiday gaps — just report
oil_gaps = panel_raw.filter(pl.col("DCOILWTICO").is_null())[["date", "DCOILWTICO"]]
print(f"  DCOILWTICO null rows : {len(oil_gaps)}")
if len(oil_gaps) > 0:
    print(oil_gaps.head(5))

In [ ]:
# ── Check 3: Calendar Alignment + Missing-Value Heatmap ──────────────────────
print("═" * 60); print("CHECK 3 — Calendar Alignment"); print("═" * 60)

dates_pd = pd.to_datetime(panel_raw["date"].to_list())
weekdays = dates_pd.dayofweek
assert (weekdays >= 5).sum() == 0, "Weekend dates found in index!"
print(f"✓  No weekend dates in spine  ({len(dates_pd)} rows)")

# Interactive missing-value heatmap (sampled every 250 rows for readability)
_cols    = panel_raw.columns[1:]
_dates   = panel_raw["date"].to_list()
_step    = 250
_tick_idx = list(range(0, len(_dates), _step))

null_matrix = np.array(
    [[panel_raw[c].is_null()[i] for c in _cols] for i in range(len(_dates))],
    dtype=np.int8,
)

fig = px.imshow(
    null_matrix.T,
    y=list(_cols),
    labels={"x": "Date", "y": "Series", "color": "Is Null"},
    color_continuous_scale=["#f0f0f0", "#d62728"],
    aspect="auto",
    title="Missing-Value Heatmap  (white = present, red = null)",
)
fig.update_xaxes(
    tickvals=_tick_idx,
    ticktext=[str(_dates[i]) for i in _tick_idx],
    tickangle=45,
)
fig.show()

In [ ]:
# ── Check 4: Data Types ───────────────────────────────────────────────────────
print("═" * 60); print("CHECK 4 — Data Types"); print("═" * 60)

assert panel_raw["date"].dtype == pl.Date, f"date dtype={panel_raw['date'].dtype}"
for col in panel_raw.columns[1:]:
    assert panel_raw[col].dtype == pl.Float64, (
        f"{col} dtype={panel_raw[col].dtype}, expected Float64"
    )

print("✓  date column         : pl.Date")
print("✓  All 12 numeric cols : pl.Float64")
print()
print(panel_raw.schema)

In [ ]:
# ── Check 5: Sanity Values ────────────────────────────────────────────────────
print("═" * 60); print("CHECK 5 — Sanity Values"); print("═" * 60)

vix = panel_raw["VIXCLS"].drop_nulls()
assert (vix >= 8).all() and (vix <= 90).all(), (
    f"VIX out of [8, 90]: min={vix.min():.2f}  max={vix.max():.2f}"
)
print(f"✓  VIX range : [{vix.min():.2f}, {vix.max():.2f}]  (within [8, 90])")

gspc = panel_raw["^GSPC"].drop_nulls()
assert (gspc > 0).all(), "^GSPC has non-positive values!"
print(f"✓  ^GSPC always positive  : min={gspc.min():.2f}")

hy = panel_raw["BAMLH0A0HYM2"].drop_nulls()
assert (hy > 0).all(), f"BAMLH0A0HYM2 has non-positive values: {hy.min()}"
print(f"✓  BAMLH0A0HYM2 range : [{hy.min():.2f}, {hy.max():.2f}]")

for col in ["DGS10", "DTB3"]:
    below = panel_raw.filter(pl.col(col) < -2).select(["date", col])
    if len(below) > 0:
        print(f"  ⚠️  {col}: {len(below)} values below -2:\n{below}")
    else:
        print(f"✓  {col}: no values below -2")

# April 2020 negative oil price — must be preserved
wti_apr2020 = (
    panel_raw
    .filter(
        (pl.col("date") >= pl.lit("2020-04-20").str.to_date()) &
        (pl.col("date") <= pl.lit("2020-04-22").str.to_date())
    )
    .select(["date", "DCOILWTICO"])
)
print(f"\n  WTI around April 2020 negative-price event:\n{wti_apr2020}")
assert panel_raw.filter(pl.col("DCOILWTICO") < 0).shape[0] >= 1, (
    "April 2020 negative WTI value is missing — do not drop!"
)
print("✓  Negative WTI value preserved (real market event)")

## 5. Transformations

In [ ]:
# ── 5a. Forward-fill monthly macro, then log-diff prices, diff rates ──────────
panel_t = (
    panel_raw
    .sort("date")
    # Forward-fill monthly series (repeat last known value on non-release days)
    .with_columns([pl.col(c).forward_fill() for c in FRED_MONTHLY])
    # Log-returns for price series
    .with_columns([pl.col(c).log(base=math.e).diff().alias(c) for c in PRICE_COLS])
    # First-difference for rate series (level change in percentage points)
    .with_columns([pl.col(c).diff().alias(c) for c in RATE_COLS])
)

for c in FRED_MONTHLY:
    n_null = panel_t[c].null_count()
    assert n_null == 0, f"{c} still has {n_null} nulls after forward-fill"

print("✓  Monthly macro forward-filled  (null_count = 0 for each)")
print(f"panel_transformed shape : {panel_t.shape}")
panel_t.head(3)

In [ ]:
# ── Check 6: Transformation Verification ─────────────────────────────────────
print("═" * 60); print("CHECK 6 — Transformation Verification"); print("═" * 60)

# Monthly macro sample — should repeat the same value for ~21 trading days
sample = (
    panel_t
    .filter(
        (pl.col("date") >= pl.lit("2019-01-01").str.to_date()) &
        (pl.col("date") <= pl.lit("2019-03-31").str.to_date())
    )
    .select(["date"] + FRED_MONTHLY)
)
print("Monthly macro forward-fill sample (should see repeated values):")
print(sample.head(10))

# First log-return must be null (no t-1 for the first row)
first_ret = panel_t["^GSPC"][0]
assert first_ret is None or panel_t["^GSPC"].is_null()[0], (
    "First ^GSPC log-return should be null — found non-null value"
)
print("\n✓  First log-return is null")

# Extreme returns — very few rows should exceed |25%| (only 1987 Black Monday, COVID-level events)
gspc_ret = panel_t["^GSPC"].drop_nulls()
extreme_pct = (gspc_ret.abs() > 0.25).sum() / len(gspc_ret) * 100
print(f"✓  |^GSPC daily log-return| > 25%: {extreme_pct:.4f}% of rows (expect ≈ 0)")

# HY OAS spike during 2008 crisis
hy_oct08 = (
    panel_t
    .filter(
        (pl.col("date") >= pl.lit("2008-09-01").str.to_date()) &
        (pl.col("date") <= pl.lit("2008-12-31").str.to_date())
    )
    .select(["date", "BAMLH0A0HYM2"])
)
print(f"\nLargest HY OAS daily moves Sep–Dec 2008 (crisis spike):")
print(hy_oct08.sort("BAMLH0A0HYM2", descending=True).head(5))

## 6. Interactive Visualisations (Plotly)

In [ ]:
# ── 6. Plotly interactive charts ─────────────────────────────────────────────
_series_cols = panel_raw.columns[1:]   # 12 series (same order as panel_raw)

# ── 6a. 12-panel raw-level overview ──────────────────────────────────────────
fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=list(_series_cols),
    shared_xaxes=False,
)
for i, col in enumerate(_series_cols):
    row, col_idx = divmod(i, 3)
    s = panel_raw.filter(pl.col(col).is_not_null()).select(["date", col])
    fig.add_trace(
        go.Scatter(
            x=s["date"].to_list(), y=s[col].to_list(),
            name=col, line=dict(width=1), showlegend=False,
        ),
        row=row + 1, col=col_idx + 1,
    )
fig.update_layout(height=900, title_text="All 12 Series — Raw Levels (1990–2025)")
fig.show()

# ── 6b. VIX regime chart ─────────────────────────────────────────────────────
_vix = panel_raw.filter(pl.col("VIXCLS").is_not_null()).select(["date", "VIXCLS"])
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=_vix["date"].to_list(), y=_vix["VIXCLS"].to_list(),
    line=dict(color="#333", width=1), name="VIX",
))
for lo, hi, color, label in [
    (0,  20,  "rgba(0,200,0,0.12)",   "Low (<20)"),
    (20, 30,  "rgba(255,200,0,0.18)", "Elevated (20–30)"),
    (30, 100, "rgba(200,0,0,0.15)",   "High (>30)"),
]:
    fig2.add_hrect(
        y0=lo, y1=hi, fillcolor=color, layer="below", line_width=0,
        annotation_text=label, annotation_position="right",
    )
fig2.update_layout(
    title="VIX Regime Chart (1990–2025)",
    yaxis_title="VIX", height=420,
)
fig2.show()

# ── 6c. Rolling 252-day vol vs VIX ───────────────────────────────────────────
_rv_src = (
    panel_t
    .filter(pl.col("^GSPC").is_not_null())
    .select(["date", "^GSPC", "VIXCLS"])
    .to_pandas()
    .set_index("date")
)
_rv_src["realized_vol"] = _rv_src["^GSPC"].rolling(252).std() * np.sqrt(252) * 100

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Scatter(
    x=_rv_src.index, y=_rv_src["realized_vol"],
    name="Realized Vol 252d (%)", line=dict(width=1.5),
), secondary_y=False)
fig3.add_trace(go.Scatter(
    x=_rv_src.index, y=_rv_src["VIXCLS"],
    name="VIX", line=dict(dash="dot", width=1),
), secondary_y=True)
fig3.update_layout(title="Rolling 252-day Realized Vol vs VIX", height=400)
fig3.show()

## 7. Static Figures (Matplotlib)

In [ ]:
# ── 7. Matplotlib static figures ─────────────────────────────────────────────
import os; os.makedirs("figures", exist_ok=True)

_ret = panel_t["^GSPC"].drop_nulls().to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── 7a. Log-returns histogram vs Gaussian ────────────────────────────────────
ax = axes[0]
ax.hist(_ret, bins=120, density=True, color="#4C72B0", alpha=0.7, label="Empirical")
_mu, _sigma = _ret.mean(), _ret.std()
_x = np.linspace(_ret.min(), _ret.max(), 400)
ax.plot(_x, spstats.norm.pdf(_x, _mu, _sigma), "r-", lw=2,
        label=f"N({_mu:.4f}, {_sigma:.4f})")
ax.set_title("^GSPC Daily Log-Returns:\nEmpirical vs Normal")
ax.set_xlabel("Log-return"); ax.legend()

# ── 7b. Correlation heatmap ───────────────────────────────────────────────────
ax = axes[1]
_num_cols = list(panel_t.columns[1:])
_corr = panel_t.select(_num_cols).drop_nulls().to_pandas().corr().values
im = ax.imshow(_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(_num_cols))); ax.set_xticklabels(_num_cols, rotation=90, fontsize=7)
ax.set_yticks(range(len(_num_cols))); ax.set_yticklabels(_num_cols, fontsize=7)
for i in range(len(_num_cols)):
    for j in range(len(_num_cols)):
        ax.text(j, i, f"{_corr[i,j]:.1f}", ha="center", va="center",
                fontsize=5, color="white" if abs(_corr[i, j]) > 0.6 else "black")
fig.colorbar(im, ax=ax)
ax.set_title("Correlation Matrix\n(transformed series)")

# ── 7c. Cumulative returns & drawdown ─────────────────────────────────────────
ax = axes[2]
_cum_ret   = np.cumsum(np.nan_to_num(_ret))
_run_max   = np.maximum.accumulate(_cum_ret)
_drawdown  = _cum_ret - _run_max
_dates_dd  = (
    panel_t.filter(pl.col("^GSPC").is_not_null())["date"].to_list()
)
ax.plot(_dates_dd, _cum_ret, color="#2ca02c", lw=1, label="Cumulative log-return")
ax.fill_between(_dates_dd, _cum_ret, _run_max,
                where=(_drawdown < 0), alpha=0.4, color="#d62728", label="Drawdown")
ax.set_title("^GSPC Cumulative Returns & Drawdown")
ax.set_xlabel("Date"); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("figures/sp500_analysis.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/sp500_analysis.pdf")

## 8. Quantitative Metrics

In [ ]:
# ── 8. Quantitative Metrics ───────────────────────────────────────────────────
TRADING_DAYS = 252

quant_results = []
for col in panel_t.columns[1:]:
    s = panel_t[col].drop_nulls().to_numpy()
    if len(s) < 50:
        continue
    ann_mean = s.mean() * TRADING_DAYS
    ann_vol  = s.std()  * np.sqrt(TRADING_DAYS)
    sharpe   = ann_mean / ann_vol if ann_vol > 0 else np.nan
    cum      = np.cumsum(s)
    run_max  = np.maximum.accumulate(cum)
    max_dd   = (cum - run_max).min()
    quant_results.append({
        "series":       col,
        "ann_mean":     round(ann_mean,  4),
        "ann_vol":      round(ann_vol,   4),
        "sharpe":       round(sharpe,    4),
        "max_dd":       round(max_dd,    4),
        "skew":         round(spstats.skew(s),     4),
        "excess_kurt":  round(spstats.kurtosis(s), 4),
    })

metrics_df = pl.DataFrame(quant_results)
print(metrics_df.to_pandas().to_string(index=False))

# ── VaR & CVaR for ^GSPC ─────────────────────────────────────────────────────
_gspc_ret = panel_t["^GSPC"].drop_nulls().to_numpy()
var95  = np.percentile(_gspc_ret, 5)
cvar95 = _gspc_ret[_gspc_ret <= var95].mean()
print(f"\n^GSPC daily VaR(95%)  = {var95:.4f}")
print(f"^GSPC daily CVaR(95%) = {cvar95:.4f}")

# ── Calmar ratio (annualised return / |max drawdown|) for ^GSPC ────────────
_gspc_ann  = _gspc_ret.mean() * TRADING_DAYS
_gspc_cum  = np.cumsum(_gspc_ret)
_gspc_mdd  = (_gspc_cum - np.maximum.accumulate(_gspc_cum)).min()
calmar     = _gspc_ann / abs(_gspc_mdd) if _gspc_mdd != 0 else np.nan
print(f"^GSPC Calmar ratio    = {calmar:.4f}")

## 9. Statistical Analysis

In [ ]:
# ── 9. Statistical Analysis ───────────────────────────────────────────────────
print("═" * 60); print("STATISTICAL ANALYSIS"); print("═" * 60)

_gspc_ret = panel_t["^GSPC"].drop_nulls().to_numpy()

# ── 9a. Jarque-Bera normality test ────────────────────────────────────────────
jb_stat, jb_p = spstats.jarque_bera(_gspc_ret)
print(f"\nJarque-Bera (^GSPC log-returns):")
print(f"  stat={jb_stat:.2f}  p={jb_p:.2e}  → {'Non-normal ✓' if jb_p < 0.05 else 'Cannot reject normality'}")

# ── 9b. ADF unit-root test for every transformed series ──────────────────────
print("\nADF Stationarity Tests (H₀: unit root):")
adf_rows = []
for col in panel_t.columns[1:]:
    s = panel_t[col].drop_nulls().to_numpy()
    if len(s) < 30:
        continue
    _adf = adfuller(s, autolag="AIC")
    adf_rows.append({
        "series":         col,
        "adf_stat":       round(_adf[0], 4),
        "p_value":        round(_adf[1], 4),
        "is_stationary":  _adf[1] < 0.05,
    })
adf_df = pl.DataFrame(adf_rows)
print(adf_df.to_pandas().to_string(index=False))

# ── 9c. Ljung-Box (ARCH effects) on squared ^GSPC returns ────────────────────
lb = acorr_ljungbox(_gspc_ret ** 2, lags=[10, 20], return_df=True)
print(f"\nLjung-Box on squared returns (ARCH-effect test):\n{lb}")

# ── 9d. Granger causality: BAMLH0A0HYM2 → ^GSPC ─────────────────────────────
_hy_gspc = (
    panel_t
    .select(["date", "BAMLH0A0HYM2", "^GSPC"])
    .drop_nulls()
    .to_pandas()
)
print("\nGranger Causality: BAMLH0A0HYM2 → ^GSPC (lags 1–5):")
gc_res = grangercausalitytests(_hy_gspc[["^GSPC", "BAMLH0A0HYM2"]], maxlag=5, verbose=False)
for lag, res in gc_res.items():
    f_stat = res[0]["ssr_ftest"][0]
    p_val  = res[0]["ssr_ftest"][1]
    star   = "*" if p_val < 0.05 else ""
    print(f"  lag={lag}: F={f_stat:.3f}  p={p_val:.4f} {star}")

# ── 9e. Rolling 60-day correlation: ^GSPC vs VIXCLS ─────────────────────────
_gv = (
    panel_t
    .filter(pl.col("^GSPC").is_not_null() & pl.col("VIXCLS").is_not_null())
    .select(["date", "^GSPC", "VIXCLS"])
    .to_pandas()
    .set_index("date")
)
_gv["roll_corr"] = _gv["^GSPC"].rolling(60).corr(_gv["VIXCLS"])
fig = px.line(
    _gv.reset_index(), x="date", y="roll_corr",
    title="Rolling 60-day Correlation: ^GSPC vs VIXCLS",
    labels={"roll_corr": "Pearson r", "date": "Date"},
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.update_layout(height=380)
fig.show()

## 10. Output & Final Check

In [ ]:
# ── 10. Serialise to Parquet, upload to S3, verify round-trip (Check 7) ───────
today_str   = date.today().strftime("%Y%m%d")
parquet_name = f"panel_1990_2025_{today_str}.parquet"

# Serialise
buf = io.BytesIO()
panel_t.write_parquet(buf)
buf.seek(0)
parquet_bytes  = buf.getvalue()
file_size_mb   = len(parquet_bytes) / 1_048_576
print(f"Parquet size : {file_size_mb:.3f} MB")
assert 0.5 <= file_size_mb <= 10, (
    f"File size {file_size_mb:.2f} MB outside expected [0.5, 10] MB"
)

# Upload
s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY"),
    endpoint_url=os.environ.get("AWS_ENDPOINT_URL"),   # supports LocalStack
)
s3_key = f"{S3_PREFIX}{parquet_name}"
s3.put_object(
    Bucket=S3_BUCKET, Key=s3_key,
    Body=parquet_bytes,
    ContentType="application/octet-stream",
)
print(f"Uploaded     : s3://{S3_BUCKET}/{s3_key}")

# ── Check 7: round-trip schema & shape verify ─────────────────────────────────
print("\n═" * 30 + "\nCHECK 7 — S3 Round-trip\n" + "═" * 30)
resp = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
panel_rb = pl.read_parquet(io.BytesIO(resp["Body"].read()))

assert panel_rb.shape == panel_t.shape, (
    f"Shape mismatch: read-back {panel_rb.shape} vs written {panel_t.shape}"
)
assert panel_rb["date"].dtype == pl.Date, (
    f"date dtype after read-back: {panel_rb['date'].dtype}"
)
for col in panel_t.columns[1:]:
    assert panel_rb[col].dtype == pl.Float64, (
        f"{col} dtype mismatch after read-back: {panel_rb[col].dtype}"
    )

print(f"✓  Shape          : {panel_rb.shape}")
print(f"✓  date dtype     : {panel_rb['date'].dtype}")
print(f"✓  All 12 numeric : pl.Float64")
print(f"✓  File size      : {file_size_mb:.3f} MB  (within [0.5, 10] MB)")
print(f"\nAll 7 checks passed. Panel ready for modelling.\n")
print(f"Output : s3://{S3_BUCKET}/{s3_key}")